To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel  # FastVisionModel for LLMs
import torch
max_seq_length = 4096
load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",  # Llama-3.1 2x faster
    "unsloth/Mistral-Small-Instruct-2409",  # Mistral 22b 2x faster!
    "unsloth/Phi-4",  # Phi-4 2x faster!
    "unsloth/Phi-4-unsloth-bnb-4bit",  # Phi-4 Unsloth Dynamic 4-bit Quant
    "unsloth/gemma-2-9b-bnb-4bit",  # Gemma 2x faster!
    "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",  # Qwen 2.5 2x faster!
    "unsloth/Llama-3.2-1B-bnb-4bit",  # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
]  # More models at https://unsloth.ai/docs/get-started/all-our-models

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-4",
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.4 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the `Phi-4` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. But we convert it to HuggingFace's normal multiturn format `("role", "content")` instead of `("from", "value")`/ Phi-4 renders multi turn conversations like below:

```
<|im_start|>user<|im_sep|>Hello!<|im_end|>
<|im_start|>assistant<|im_sep|>Hi! How can I help?<|im_end|>
<|im_start|>user<|im_sep|>What is 2+2?<|im_end|>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, phi4, llama3` and more.

In [ ]:
from datasets import load_dataset
# 1. Define the template with explicit newline management
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}


### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Clean the input: Strip trailing whitespace which often causes token bleeding
        cleaned_input = input_text.strip()

        # Smart Truncation
        if len(cleaned_input) > 3000:
            cleaned_input = cleaned_input[:1500] + "\n[TRUNCATED]\n" + cleaned_input[-1500:]

        # Format: Note the double newline before Response
        text = alpaca_prompt.format(instruction, cleaned_input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }
dataset = load_dataset("Jaiccc/Terminal_Recording_Group_Boundary_Timestamp", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
print("=== THE RAW OUTPUT COLUMN ===")
print(dataset[1]['output'])
# This should print ONLY the timestamp (e.g., 21.987222)

print("\n=== THE FORMATTED TEXT COLUMN (What the model actually sees) ===")
print(dataset[1]['text']) # Printing just the first 500 characters
# This should start with "Below is an instruction..." followed by your XML.

=== THE RAW OUTPUT COLUMN ===
37.178971

=== THE FORMATTED TEXT COLUMN (What the model actually sees) ===
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a strict data extraction algorithm. Your purpose is to output ONLY the numerical timestamp of the next command event in the XML log.

### DEFINITION OF A BOUNDARY:
A boundary is the **first `<user_input>` tag following any system-provided state where the terminal is awaiting input.**
An input state includes:
1. **Shell Prompts:** Lines containing `$` or `#` (e.g., `user@host:~$`).
2. **Authentication:** Prompts asking for secrets (e.g., `password for`, `[sudo]`).
3. **Interactive Pagers:** Terminal states indicating a pause or waiting for user interaction (e.g., `(END)`, ``, or prompts lacking a trailing newline).

### YOUR ALGORITHM:
1. **START POINT:** The current cursor is at timestamp: 21.987222.


In [ ]:
def manual_labeling_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]

    all_input_ids = []
    all_labels = []

    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # 1. Clean and truncate input as before
        cleaned_input = input_text.strip()
        if len(cleaned_input) > 3000:
            cleaned_input = cleaned_input[:1500] + "\n[TRUNCATED]\n" + cleaned_input[-1500:]

        # 2. Build the two parts separately
        prompt_part = f"### Instruction:\n{instruction}\n\n### Input:\n{cleaned_input}\n\n### Response: "
        response_part = f"{output}{tokenizer.eos_token}"

        # 3. Tokenize them separately
        prompt_ids = tokenizer.encode(prompt_part, add_special_tokens=True)
        response_ids = tokenizer.encode(response_part, add_special_tokens=False)

        # 4. Create the labels
        # Mask the prompt part with -100 (ignore)
        # Keep the response part as the actual token IDs (learn)
        input_ids = prompt_ids + response_ids
        labels = ([-100] * len(prompt_ids)) + response_ids

        # 5. Handle truncation to max_seq_length
        all_input_ids.append(input_ids[:max_seq_length])
        all_labels.append(labels[:max_seq_length])

    return {
        "input_ids": all_input_ids,
        "labels": all_labels,
    }

# Apply the manual labeling
# NOTE: We remove the "text" field to ensure the trainer uses our manual labels
dataset = dataset.map(manual_labeling_func, batched = True, remove_columns = dataset.column_names)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    # IMPORTANT: We don't use dataset_text_field because we provided labels manually
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# NO "train_on_responses_only" call here. It is no longer needed.

We verify masking is actually done:

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'### Instruction:\nYou are a strict data extraction algorithm. Your purpose is to output ONLY the numerical timestamp of the next command event in the XML log.\n\n### DEFINITION OF A BOUNDARY:\nA boundary is the **first `<user_input>` tag following any system-provided state where the terminal is awaiting input.**\nAn input state includes:\n1. **Shell Prompts:** Lines containing `$` or `#` (e.g., `user@host:~$`).\n2. **Authentication:** Prompts asking for secrets (e.g., `password for`, `[sudo]`).\n3. **Interactive Pagers:** Terminal states indicating a pause or waiting for user interaction (e.g., `(END)`, ``, or prompts lacking a trailing newline).\n\n### YOUR ALGORITHM:\n1. **START POINT:** The current cursor is at timestamp: 50.572229.\n2. **SCAN:** Begin reading the log sequentially, looking only at `<system_output>` tags that occur AFTER 50.572229.\n3. **IDENTIFY STATE:** Determine if the system is waiting for input based on the definitions above.\n4. **EXTRACT:** Identify the very 

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

We can see the System and Instruction prompts are successfully masked!

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.251 GB.
13.227 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 3 | Total steps = 6
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 65,536,000 of 14,725,043,200 (0.45% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,2.117400
2,5.354700
3,1.923800
4,5.965000
5,2.068100
6,2.328400


wandb: WARNING URL not available in offline run


train/epoch,▁▂▄▅▇██
train/global_step,▁▂▄▅▇██
train/grad_norm,▄▅▄▅▁█
train/learning_rate,▁▂▄▅▇█
train/loss,▁▇▁█▁▂
total_flos,3444122669137920.0
train/epoch,3
train/global_step,6
train/grad_norm,7.65533
train/learning_rate,0.0002
train/loss,2.3284


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

124.8487 seconds used for training.
2.08 minutes used for training.
Peak reserved memory = 28.273 GB.
Peak reserved memory for training = 15.046 GB.
Peak reserved memory % of max memory = 35.675 %.
Peak reserved memory for training % of max memory = 18.985 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`.

In [37]:
from datasets import load_dataset
from unsloth import FastLanguageModel
import torch
import re

# 1. Enable fast inference
FastLanguageModel.for_inference(model)

# 2. Load the specific dataset and select a sample (matching notebook index 7)
dataset = load_dataset("Jaiccc/Terminal_Recording_Group_Boundary_Timestamp", split="train")
sample = dataset[1]

# Extract components from the dataset
instruction = sample['instruction']
input_data = sample['input']
expected_output = sample['output']
print(input_data)
# Combine instruction and input into a single block
combined_user_text = f"{instruction}\n\n{input_data}"

# 3. Format the prompt to match ChatML markers
# Note: Ensure your tokenizer was initialized with the 'phi-4' template
prompt = f"<|im_start|>user<|im_sep|>{combined_user_text}<|im_end|><|im_start|>assistant<|im_sep|>"

# 4. Tokenize and move to GPU
inputs = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# 5. Generate output using notebook parameters
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.2, # As specified in notebook code
)

# 6. Decode the raw output
raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

# 7. Extract the assistant response using regex
m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
if m:
    prediction = m.group(1).strip()
else:
    # Fallback: find text after the last assistant marker
    prediction = raw_output.split("<|im_start|>assistant<|im_sep|>")[-1].replace("<|im_end|>", "").strip()

print("=== ROW 7 TEST RESULTS ===")
print(f"EXPECTED: {expected_output}")
print(f"GENERATED: {prediction}")

<user_input timestamp="21.987222">c</user_input>
<system_output timestamp="21.989407">c</system_output>
<user_input timestamp="22.208425">d</user_input>
<system_output timestamp="22.213483">d</system_output>
<user_input timestamp="22.309141"> </user_input>
<system_output timestamp="22.31413"> </system_output>
<user_input timestamp="22.568401">/</user_input>
<system_output timestamp="22.585127">/</system_output>
<user_input timestamp="22.915005">h</user_input>
<system_output timestamp="22.936477">h</system_output>
<user_input timestamp="22.996323">o</user_input>
<system_output timestamp="22.998952">o</system_output>
<user_input timestamp="23.419145">m</user_input>
<system_output timestamp="23.428202">m</system_output>
<user_input timestamp="23.60239">e</user_input>
<system_output timestamp="23.614747">e</system_output>
<user_input timestamp="23.907324">/</user_input>
<system_output timestamp="23.927054">/</system_output>
<user_input timestamp="24.251554">f</user_input>
<system_output ti

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("phi_lora")  # Local saving
tokenizer.save_pretrained("phi_lora")
# model.push_to_hub("your_name/phi_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/phi_lora", token = "YOUR_HF_TOKEN") # Online saving

('phi_lora/tokenizer_config.json',
 'phi_lora/special_tokens_map.json',
 'phi_lora/chat_template.jinja',
 'phi_lora/vocab.json',
 'phi_lora/merges.txt',
 'phi_lora/added_tokens.json',
 'phi_lora/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "phi_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Describe a tall tower in the capital of France."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(
    input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
    use_cache = True, temperature = 1.5, min_p = 0.1
)

One of the most iconic tall towers in the capital of France, Paris, is the Eiffel Tower. Standing at approximately 324 meters (1,063 feet) tall, it is one of the most recognizable structures in the world. The Eiffel Tower was designed by the engineer Gustave Eiffel and his collaborators for the 1889 Exposition Universelle (World's Fair) to celebrate the 100th anniversary of the French Revolution.

The tower is constructed of wrought iron and consists of three levels accessible to visitors. The first and second levels offer observation decks with stunning panoramic views of Paris, while the third level,


You can also use Hugging Face's `AutoPeftModelForCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer

    model = AutoPeftModelForCausalLM.from_pretrained(
        "phi_lora",  # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("phi_lora")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("phi_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/phi_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("phi_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/phi_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("phi_lora")
    tokenizer.save_pretrained("phi_lora")
if False:
    model.push_to_hub("HF_USERNAME/phi_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/phi_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list in our [Docs](https://unsloth.ai/docs/basics/saving-and-using-models/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("phi_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/phi_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("phi_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/phi_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("phi_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/phi_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/phi_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )